# Logger Engine

Esta é uma documentação detalhada e didática da classe LoggerEngine, projetada para ser o núcleo de rastreabilidade e monitoramento de aplicações Python.

## Visão Geral

A LoggerEngine não é apenas um simples formatador de mensagens; ela é um orquestrador de telemetria. Sua principal função é centralizar a captura de eventos do sistema e decidir, com base em configurações dinâmicas (variáveis de ambiente ou parâmetros de instância), para onde esses dados devem ir: console, arquivos locais ou um banco de dados NoSQL (MongoDB).

O objetivo principal é garantir que o desenvolvedor tenha logs estruturados. Em vez de simples linhas de texto, a classe transforma cada evento em um objeto de dados rico, facilitando auditorias e a busca por erros em ambientes de produção.

## Fluxo de Execução

Para entender como a classe opera, imagine o seguinte caminho percorrido por uma informação:

1. **Chamada do Método `log()`**: O usuário envia a mensagem, o nível (INFO, ERROR, etc.) e metadados.
2. **Resolução de Configurações**: O motor verifica se deve seguir a configuração global da classe ou se houve uma instrução específica para aquela chamada (ex: forçar o salvamento no MongoDB apenas para um erro crítico).
3. **Construção do Payload**: Os dados brutos são enviados para um "Builder" que padroniza o formato (adiciona timestamps, IDs únicos e formata o JSON).
4. **Configuração do Logger**: A classe prepara os "Handlers" (saídas). Se `save_logs` for True, ela abre um canal com o arquivo; se for exibir no console, define o nível de verbosidade.
5. **Emissão e Persistência**: A mensagem é exibida na tela e, simultaneamente, o payload estruturado é enviado para o MongoDB.

## Resumo dos Métodos

| **Método** | **Responsabilidade** |
| --- | --- |
| `_get_env_bool` | Lê variáveis de ambiente (OS) e as converte para booleano com segurança. |
| `_resolve_config` | Decide quais flags (salvar, mostrar, persistir) serão usadas na execução atual. |
| `_build_payloads` | Transforma os argumentos em mensagens amigáveis e objetos JSON estruturados. |
| `_get_logger` | Configura a biblioteca nativa `logging` do Python (Handlers e Formatters). |
| `_emit_log` | Realiza o disparo final da mensagem para as saídas de texto. |
| `_save_to_mongo` | Gerencia a comunicação e inserção dos logs no banco de dados. |
| `log` | **Ponto de entrada único.** Orquestra a execução de todos os métodos acima. |

## Arquitetura e Insights

- **Configuração Híbrida**: A classe utiliza um padrão de prioridade: `Variável de Ambiente` > `Argumento do Método` > `Valor Padrão da Instância`. Isso permite mudar o comportamento do log sem alterar o código, apenas mudando o ambiente.
- **Desacoplamento de Payloads**: Note que a classe diferencia a "mensagem de texto" (para humanos lerem no console) do "payload de dados" (para máquinas processarem no MongoDB).
- **Rastreabilidade Única**: O uso de um `log_id` gerado automaticamente permite que você rastreie uma transação específica através de múltiplos arquivos ou coleções de banco de dados.

## Detalhamento Técnico

---

## Classe LoggerEngine

**Descrição**
Gerenciador centralizado de logs estruturados. Permite o controle dinâmico da verbosidade e persistência de dados em múltiplos destinos, garantindo padronização em toda a aplicação.

**Argumentos**

- `log_id` (str): Identificador único da transação.
- `flag` (str): Identificador da origem (ex: "AuthService").
- `file_name` (str): Nome do módulo Python de origem.
- `log_file_name` (str): Prefixo do arquivo de log local.
- `show_info_logs` (bool): Habilita/Desabilita nível INFO no console.
- `show_metadata` (bool): Define se dados extras aparecem no texto do log.
- `save_logs` (bool): Habilita escrita em arquivo `.log`.
- `save_mongo` (bool): Habilita persistência no MongoDB.
- `format_metadata` (bool): Aplica formatação visual aos metadados.

## 1. Método log()

**Descrição**
Método principal e público. Ele serve como o "maestro" da classe, recebendo os dados e acionando os métodos privados na ordem correta para processar o log.

**Argumentos**

- `level` (str): Nível de severidade (ex: "INFO", "WARNING", "ERROR", "CRITICAL").
- `func_name` (str, opcional): Nome da função onde o log foi gerado.
- `message` (str, opcional): Texto explicativo do evento.
- `metadata` (dict, opcional): Dicionário com dados técnicos adicionais.
- `save_logs / save_mongo / show_info_logs / show_metadata` (bool, opcional): Parâmetros para sobrescrever o comportamento padrão da instância apenas para este log.

**Retornos**

- `dict`: Retorna o payload completo que foi gerado e/ou persistido.

**Raises**

- `AttributeError`: Caso o `level` informado não seja um nível de log válido.
- `ConnectionError`: Caso o salvamento no MongoDB falhe.

**Exemplos**

```bash
logger = LoggerEngine(flag="API_GATEWAY", save_mongo=True)

# Exemplo simples
logger.log(level="INFO", message="Servidor iniciado na porta 8080")

# Exemplo complexo com metadados e sobrescrita
logger.log(
    level="ERROR",
    func_name="process_payment",
    message="Falha na transação",
    metadata={"user_id": 123, "error_code": "ST-404"},
    save_logs=True # Força salvar em arquivo este erro específico
)
```

## 2. Método _get_env_bool()

**Descrição**
Utilitário interno para capturar configurações do sistema operacional de forma segura, tratando strings como "true", "1" ou "yes" como valores booleanos verdadeiros.

**Argumentos**

- `env_name` (str): O nome da chave da variável de ambiente.
- `default` (bool): O valor de retorno caso a variável não esteja definida.

**Retornos**

- `bool`: O estado lógico resolvido.

**Raises**

- `KeyError`: Se o valor presente na variável de ambiente for inválido (não conversível para booleano).

**Exemplos**

```bash
# Se no terminal: export SHOW_INFO_LOGS=True
val = engine._get_env_bool("SHOW_INFO_LOGS", False)
# val será True
```

## 3. Método _resolve_config()

**Descrição**
Este método atua como o cérebro decisório da classe. Ele compara as configurações globais (definidas ao instanciar a classe) com as configurações locais (passadas no momento do log). Se um parâmetro for enviado como `None`, ele assume o padrão da instância.

**Argumentos**

- `save_logs` (bool/None): Desejo de salvar em arquivo para este evento.
- `save_mongo` (bool/None): Desejo de persistir no banco para este evento.
- `show_info_logs` (bool/None): Desejo de mostrar nível INFO no console.
- `show_metadata` (bool/None): Desejo de exibir metadados no console.

**Retornos**

- `dict`: Um dicionário com as chaves e seus valores booleanos finais (Ex: `{'save_logs': True, ...}`).

**Exemplos**

```bash
# Instância configurada para NÃO salvar nada
engine = LoggerEngine(save_logs=False)

# Chamada forçando o salvamento
config = engine._resolve_config(save_logs=True, save_mongo=None, ...)
# config['save_logs'] será True (sobrescrita)
```

## 4. Método _build_payloads()

**Descrição**
Responsável pela padronização dos dados. Ele separa a "mensagem para humanos" (string formatada) do "payload para máquinas" (dicionário estruturado). Ele utiliza internamente uma classe auxiliar (`PayloadBuilder`) para garantir que todos os logs do sistema tenham a mesma cara.

**Argumentos**

- `level` (str): Nível do log.
- `func_name` (str): Nome da função de origem.
- `message` (str): Texto da mensagem.
- `metadata` (dict): Dados extras.
- `show_metadata` (bool): Define se o dicionário de metadados deve ser injetado na string de texto.

**Retornos**

- `tuple`: `(display_message: str, mongo_payload: dict)`

## 5. Método _get_logger()

**Descrição**
Configura a infraestrutura do Python Standard Library (`logging`). Ele cria o objeto que realmente "fala" com o console e com o sistema de arquivos, definindo cores, formatos de data e para qual arquivo o texto deve ser enviado.

**Argumentos**

- `save_logs` (bool): Se True, adiciona um `FileHandler`.
- `show_info_logs` (bool): Define o `level` mínimo do logger (DEBUG, INFO ou WARNING).

**Retornos**

- `logging.Logger`: O objeto logger pronto para uso.

## 6. Método _emit_log()

**Descrição**
É o gatilho final. Após tudo estar configurado e a mensagem formatada, este método identifica o nível solicitado (info, error, debug) e dispara o comando para o objeto logger.

**Argumentos**

- `logger` (logging.Logger): A instância configurada no método anterior.
- `level` (str): A severidade.
- `message` (str): A mensagem final já formatada.

**Raises**

- `AttributeError`: Se você tentar usar um nível que não existe (ex: `logger.log(level="BATATA")`).

## 7. Método _save_to_mongo()

**Descrição**
Realiza a persistência persistente. Ele pega o payload estruturado e o envia para uma coleção no MongoDB. Isso é crucial para dashboards futuros (como Grafana ou Kibana).

**Argumentos**

- `mongo_metadata` (dict): O payload completo gerado pelo `_build_payloads`.

**Retornos**

- `str`: O ID do documento inserido no banco de dados.

**Raises**

- `ConnectionError`: Se o banco de dados estiver fora do ar ou as credenciais estiverem incorretas.

### Resumo do Ciclo de Vida do Log

Para visualizar como esses métodos interagem, imagine esta sequência:

1. **User** chama `log()`.
2. `log()` chama `_resolve_config()` para saber as regras.
3. `log()` chama `_build_payloads()` para preparar os dados.
4. `log()` chama `_get_logger()` para preparar a saída.
5. `log()` chama `_emit_log()` para imprimir na tela/arquivo.
6. `log()` chama `_save_to_mongo()` para guardar no banco.

In [ ]:
```bash

```